# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendimos los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí experimentamos con las rutinas de limpieza.

| Paso | Descripción |
|------|-------------|
| **1** | Solventar problemas de calidad (completitud, consistencia, precisión, sensibilidad) |
| **2** | Codificación de variable categórica `ocean_proximity` |
| **3** | Feature Engineering: nuevas variables combinadas |
| **4** | Escalado de variables numéricas |

---

> ⚠️ **Regla anti data leakage:**  
> Toda estadística (mediana, parámetros del scaler, etc.) se calcula **solo sobre `train_set`**  
> y luego se aplica igual a `val_set` y `test_set`.

---
## 0. Setup

In [21]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_PATH   = PROJECT_ROOT / 'data' / 'interim'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 15)

print(f'PROJECT_ROOT   : {PROJECT_ROOT}')
print(f'INTERIM_PATH   : {INTERIM_PATH}')
print(f'PROCESSED_PATH : {PROCESSED_PATH}')

PROJECT_ROOT   : C:\Proyecto\ds-ml-project-template
INTERIM_PATH   : C:\Proyecto\ds-ml-project-template\data\interim
PROCESSED_PATH : C:\Proyecto\ds-ml-project-template\data\processed


---
## 1. Carga de los Conjuntos de Datos

In [22]:
train = pd.read_csv(INTERIM_PATH / 'train_set.csv')
val   = pd.read_csv(INTERIM_PATH / 'val_set.csv')
test  = pd.read_csv(INTERIM_PATH / 'test_set.csv')

print(f'train : {train.shape}')
print(f'val   : {val.shape}')
print(f'test  : {test.shape}')

# Separar features y target ANTES de cualquier transformación
TARGET = 'median_house_value'

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val,   y_val   = val.drop(columns=[TARGET]),   val[TARGET]
X_test,  y_test  = test.drop(columns=[TARGET]),  test[TARGET]

print(f'\nX_train : {X_train.shape}  |  y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}   |  y_val   : {y_val.shape}')
print(f'X_test  : {X_test.shape}  |  y_test  : {y_test.shape}')

train : (13209, 10)
val   : (3303, 10)
test  : (4128, 10)

X_train : (13209, 9)  |  y_train : (13209,)
X_val   : (3303, 9)   |  y_val   : (3303,)
X_test  : (4128, 9)  |  y_test  : (4128,)


---
## 2. Solventar Problemas de Calidad

Resolvemos los cuatro problemas detectados en el EDA.
Documentamos cada decisión tomada.

### 2.1 Completitud — Valores Faltantes en `total_bedrooms`

**Hallazgo:** `total_bedrooms` tiene ~207 NaN en el dataset completo (~157 en train, ~1%).

**Decisión: KNNImputer**  
Imputar con medidas de tendencia central (media, mediana, moda) no tiene sentido en producción:
- La mediana es un valor global que ignora el contexto del vecindario
- Un bloque censal con 10 hogares no debería recibir el mismo valor que uno con 500

`KNNImputer` busca los `k` registros más similares (vecinos más cercanos) y promedia sus valores.  
Es coherente con el contexto: bloques censales con características similares tendrán dormitorios similares.

> El KNNImputer se **fittea solo en train** y se aplica igual a val y test.

In [23]:
from sklearn.impute import KNNImputer

print('NaN antes de imputación:')
for df, name in [(X_train, 'train'), (X_val, 'val'), (X_test, 'test')]:
    print(f'  {name:<6}: {df.isnull().sum().sum()} NaN')

# KNNImputer fitteado SOLO sobre train
# n_neighbors=5: usa los 5 bloques censales más similares para estimar el valor
imputer = KNNImputer(n_neighbors=5)

# fit_transform en train
X_train_imputed = imputer.fit_transform(X_train.select_dtypes(include='number'))
X_train[X_train.select_dtypes(include='number').columns] = X_train_imputed

# solo transform en val y test
X_val_imputed  = imputer.transform(X_val.select_dtypes(include='number'))
X_val[X_val.select_dtypes(include='number').columns] = X_val_imputed

X_test_imputed = imputer.transform(X_test.select_dtypes(include='number'))
X_test[X_test.select_dtypes(include='number').columns] = X_test_imputed

print('\nNaN después de imputación con KNNImputer:')
for df, name in [(X_train, 'train'), (X_val, 'val'), (X_test, 'test')]:
    print(f'  {name:<6}: {df.isnull().sum().sum()} NaN ✅')

NaN antes de imputación:
  train : 138 NaN
  val   : 30 NaN
  test  : 39 NaN

NaN después de imputación con KNNImputer:
  train : 0 NaN ✅
  val   : 0 NaN ✅
  test  : 0 NaN ✅


### 2.2 Precisión — Outliers

**Hallazgo:** `total_rooms`, `total_bedrooms` y `population` tienen outliers severos (>3% fuera del rango IQR).

**Decisión:** No eliminamos los outliers en esta fase. Son valores reales de bloques censales con alta densidad (zonas urbanas de Los Ángeles, San Francisco). Eliminarlos introduciría sesgo geográfico en el modelo.

El escalado posterior (StandardScaler) reducirá su impacto en modelos lineales.
Para modelos basados en árboles (RandomForest, XGBoost), los outliers no afectan.

In [24]:
print('Verificación de outliers en train (método IQR):')
print(f'  {"Variable":<22} {"Outliers":>10} {"% total":>8}')
print('  ' + '-' * 44)
for col in X_train.select_dtypes(include=np.number).columns:
    Q1  = X_train[col].quantile(0.25)
    Q3  = X_train[col].quantile(0.75)
    lim = Q3 + 1.5 * (Q3 - Q1)
    n   = (X_train[col] > lim).sum()
    pct = n / len(X_train) * 100
    flag = '  ← se conservan (reales)' if pct > 3 else ''
    print(f'  {col:<22} {n:>10,} {pct:>7.1f}%{flag}')

Verificación de outliers en train (método IQR):
  Variable                 Outliers  % total
  --------------------------------------------
  longitude                       0     0.0%
  latitude                        0     0.0%
  housing_median_age              0     0.0%
  total_rooms                   827     6.3%  ← se conservan (reales)
  total_bedrooms                854     6.5%  ← se conservan (reales)
  population                    783     5.9%  ← se conservan (reales)
  households                    807     6.1%  ← se conservan (reales)
  median_income                 419     3.2%  ← se conservan (reales)


### 2.3 Consistencia — Variable Categórica

**Hallazgo:** `ocean_proximity` es de tipo texto con 5 categorías. Se tratará en la sección de Codificación (paso 3).

In [25]:
print('ocean_proximity — categorías en train:')
print(X_train['ocean_proximity'].value_counts())

ocean_proximity — categorías en train:
ocean_proximity
<1H OCEAN     5799
INLAND        4238
NEAR OCEAN    1694
NEAR BAY      1477
ISLAND           1
Name: count, dtype: int64


### 2.4 Sensibilidad — Censura en el Target

**Hallazgo:** `median_house_value` está censurado en $500,001. El ~1.8% de registros tiene ese valor exacto aunque el precio real sea mayor.

**Decisión:** Conservamos estos registros en el entrenamiento. Eliminarlos representaría una pérdida del 1.8% de datos y podría introducir sesgo hacia zonas costeras de alto precio. El modelo aprenderá que existe ese techo, lo cual es información válida del mercado.

In [26]:
cap_val = y_train.max()
n_cap   = (y_train == cap_val).sum()
print(f'Registros censurados en train: {n_cap} ({n_cap/len(y_train)*100:.1f}%)')
print(f'Decisión: se conservan en el entrenamiento.')

Registros censurados en train: 619 (4.7%)
Decisión: se conservan en el entrenamiento.


---
## 3. Codificación Categórica — `ocean_proximity`

Los algoritmos clásicos de ML no entienden texto. Necesitamos convertir `ocean_proximity` en números.

**Opciones:**
- **Codificación Ordinal** → asigna números enteros (1, 2, 3...). Implica un orden entre categorías.
  - Problema: ¿es `INLAND` < `NEAR BAY` < `NEAR OCEAN`? No existe ese orden lógico entre estas categorías.
- **One-Hot Encoding (Nominal)** → crea una columna binaria (0/1) por cada categoría. ✅ **decisión tomada**
  - No asume orden entre categorías
  - Correcto para variables **nominales** (sin orden natural)
  - `drop_first=True` elimina una columna para evitar multicolinealidad perfecta

> Las columnas OHE se definen sobre **train** y se replican igual en val y test.

In [27]:
# Definir columnas OHE desde train
ohe_cols = pd.get_dummies(
    X_train['ocean_proximity'], prefix='ocean', drop_first=True
).columns.tolist()
print('Columnas OHE generadas desde train:', ohe_cols)

def encode_ocean(df: pd.DataFrame, ohe_cols: list) -> pd.DataFrame:
    """Aplica One-Hot Encoding a ocean_proximity con columnas fijas."""
    df     = df.copy()
    dummies = pd.get_dummies(df['ocean_proximity'], prefix='ocean', drop_first=True)
    dummies = dummies.reindex(columns=ohe_cols, fill_value=0)
    df      = df.drop(columns=['ocean_proximity'])
    return pd.concat([df, dummies], axis=1)

X_train = encode_ocean(X_train, ohe_cols)
X_val   = encode_ocean(X_val,   ohe_cols)
X_test  = encode_ocean(X_test,  ohe_cols)

print(f'\nColumnas después del encoding ({len(X_train.columns)}):')
print(list(X_train.columns))
X_train[ohe_cols].head(3)

Columnas OHE generadas desde train: ['ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN']

Columnas después del encoding (12):
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN']


,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN
0,False,False,True,False
1,False,False,False,False
2,False,False,True,False


---
## 4. Feature Engineering — Nuevas Variables

Como se observó en el EDA, `total_rooms` no significa mucho si hay muchos hogares en un distrito.
Normalizar por hogar revela información más útil:

| Nueva variable | Fórmula | Por qué es útil |
|----------------|---------|------------------|
| `rooms_per_household` | `total_rooms / households` | Tamaño real de la vivienda promedio |
| `bedrooms_per_room` | `total_bedrooms / total_rooms` | Proporción de dormitorios vs habitaciones totales |
| `population_per_household` | `population / households` | Densidad de ocupación por hogar |

In [28]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Crea nuevas variables combinadas. No modifica el DataFrame original."""
    df = df.copy()
    df['rooms_per_household']      = df['total_rooms']    / df['households']
    df['bedrooms_per_room']        = df['total_bedrooms'] / df['total_rooms']
    df['population_per_household'] = df['population']     / df['households']
    return df

X_train = add_features(X_train)
X_val   = add_features(X_val)
X_test  = add_features(X_test)

nuevas = ['rooms_per_household', 'bedrooms_per_room', 'population_per_household']
print('Estadísticas de las nuevas features (train):')
print(X_train[nuevas].describe().round(3).to_string())

Estadísticas de las nuevas features (train):
       rooms_per_household  bedrooms_per_room  population_per_household
count           13209.0000         13209.0000                13209.0000
mean                5.4340             0.2130                    3.0090
std                 2.6110             0.0580                    4.9670
min                 0.8890             0.1000                    0.7500
25%                 4.4440             0.1750                    2.4310
50%                 5.2350             0.2030                    2.8140
75%                 6.0550             0.2400                    3.2790
max               141.9090             1.0000                  502.4620


In [29]:
# Verificar que las nuevas features mejoran la correlación con el target
# Comparar features originales vs nuevas
originales = ['total_rooms', 'total_bedrooms', 'households']

df_corr = X_train[originales + nuevas].copy()
df_corr[TARGET] = y_train.values

corr = df_corr.corr()[TARGET].drop(TARGET)

print('Correlación con median_house_value:')
print('-' * 52)
print('  Variables originales:')
for col in originales:
    print(f'    {col:<28} {corr[col]:+.4f}')
print('  Nuevas features:')
for col in nuevas:
    bar = '█' * int(abs(corr[col]) * 25)
    print(f'    {col:<28} {corr[col]:+.4f}  {bar}')

Correlación con median_house_value:
----------------------------------------------------
  Variables originales:
    total_rooms                  +0.1327
    total_bedrooms               +0.0531
    households                   +0.0685
  Nuevas features:
    rooms_per_household          +0.1436  ███
    bedrooms_per_room            -0.2551  ██████
    population_per_household     -0.0332  


---
## 5. Escalado de Variables Numéricas

**¿Por qué escalar?**  
Variables como `population` (rango 3–35,000) y `median_income` (rango 0.5–15) tienen escalas muy distintas.
Esto perjudica a algoritmos basados en distancias (KNN, SVM) y gradientes (regresión lineal, redes neuronales).

**`latitude` y `longitude` se excluyen del escalado.**  
Son coordenadas geográficas. Escalarlas no tiene sentido: no son magnitudes comparables con ingresos o habitaciones,
y su relación con el precio es espacial, no lineal.

**Opciones:**
- `StandardScaler` → media=0, std=1. Bueno cuando la distribución es aproximadamente normal. ✅ **decisión tomada**
- `MinMaxScaler` → rango [0,1]. Más sensible a outliers extremos.

Elegimos `StandardScaler` porque nuestras variables tienen outliers significativos.

> El scaler se **fittea solo en train** y se aplica (transform) en val y test.

In [30]:
from sklearn.preprocessing import StandardScaler

# Excluir coordenadas geográficas y columnas OHE del escalado
# latitude y longitude son coordenadas — escalarlas no tiene sentido semántico
EXCLUDE_FROM_SCALING = ['latitude', 'longitude'] + ohe_cols

num_cols = [c for c in X_train.columns if c not in EXCLUDE_FROM_SCALING]
print(f'Columnas a escalar ({len(num_cols)}):')
print(num_cols)
print(f'\nExcluidas del escalado: {EXCLUDE_FROM_SCALING}')

scaler = StandardScaler()

# fit_transform en train
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# solo transform en val y test (sin fit)
X_val[num_cols]  = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('\nEstadísticas después del escalado en train (media ≈ 0, std ≈ 1):')
print(X_train[num_cols].describe().loc[['mean', 'std']].round(3).to_string())

Columnas a escalar (9):
['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household']

Excluidas del escalado: ['latitude', 'longitude', 'ocean_INLAND', 'ocean_ISLAND', 'ocean_NEAR BAY', 'ocean_NEAR OCEAN']

Estadísticas después del escalado en train (media ≈ 0, std ≈ 1):
      housing_median_age  total_rooms  total_bedrooms  population  households  median_income  rooms_per_household  bedrooms_per_room  population_per_household
mean             -0.0000       0.0000          0.0000      0.0000     -0.0000        -0.0000               0.0000            -0.0000                   -0.0000
std               1.0000       1.0000          1.0000      1.0000      1.0000         1.0000               1.0000             1.0000                    1.0000


---
## 6. Verificación Final y Guardado

In [31]:
print('RESUMEN FINAL DEL PIPELINE')
print('=' * 55)
for X, name in [(X_train, 'X_train'), (X_val, 'X_val'), (X_test, 'X_test')]:
    nan   = X.isnull().sum().sum()
    obj   = (X.dtypes == object).sum()
    print(f'{name:<10}: shape={X.shape}  NaN={nan}  columnas texto={obj}')

print(f'\nColumnas finales ({len(X_train.columns)}):')
for col in X_train.columns:
    dtype = str(X_train[col].dtype)
    print(f'  {col:<30} {dtype}')

RESUMEN FINAL DEL PIPELINE
X_train   : shape=(13209, 15)  NaN=0  columnas texto=0
X_val     : shape=(3303, 15)  NaN=0  columnas texto=0
X_test    : shape=(4128, 15)  NaN=0  columnas texto=0

Columnas finales (15):
  longitude                      float64
  latitude                       float64
  housing_median_age             float64
  total_rooms                    float64
  total_bedrooms                 float64
  population                     float64
  households                     float64
  median_income                  float64
  ocean_INLAND                   bool
  ocean_ISLAND                   bool
  ocean_NEAR BAY                 bool
  ocean_NEAR OCEAN               bool
  rooms_per_household            float64
  bedrooms_per_room              float64
  population_per_household       float64


In [32]:
# Guardar los datasets procesados con el target incluido
for X, y, name in [
    (X_train, y_train, 'train'),
    (X_val,   y_val,   'val'),
    (X_test,  y_test,  'test'),
]:
    out = X.copy()
    out[TARGET] = y.values
    path = PROCESSED_PATH / f'{name}_processed.csv'
    out.to_csv(path, index=False)
    print(f'✅ {name}_processed.csv  →  {out.shape[0]:,} filas x {out.shape[1]} columnas')

print(f'\nArchivos en data/processed/:')
for f in sorted(PROCESSED_PATH.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')

✅ train_processed.csv  →  13,209 filas x 16 columnas
✅ val_processed.csv  →  3,303 filas x 16 columnas
✅ test_processed.csv  →  4,128 filas x 16 columnas

Archivos en data/processed/:
  test_processed.csv  (912.8 KB)
  train_processed.csv  (2919.9 KB)
  val_processed.csv  (730.2 KB)
